# QAOA with layer-norm parameter pre-processing

Layer-norm a parameter vector then apply via apply_linear. Same trace, two routes.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the planner picks the route (cpu / cpu+sim / cpu+gpu) at preflight time.

## Background

**Problem:** combinatorial optimization via QAOA. **Classical:** simulated annealing. **Quantum:** QAOA — alternating cost / mixer layers, expect on cost Hamiltonian.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx.ops.primitives.evolution import apply_linear
from uniqx.ops.primitives.nn import layer_norm

n = 4
rng = np.random.default_rng(2)
state = rng.standard_normal(n) * 0.5
M = rng.standard_normal((n, n)) * 0.2
U = 0.5 * (M + M.T)

@trace
def prog(vec, op):
    normed = layer_norm(vec)
    return apply_linear(op, normed)

module = prog(state.tolist(), U.tolist())


## Preflight — see what routes the planner found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the planner's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the planner picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
